# Analysis of finished ```.npz``` files

This Jupyter-Notebook should load the ```.npz```-files which contain the background-residual-field without any coil current, the field with coil current and the simulation results for the field with coil current. *Note* that ```.npz``` files (numpy-zip) need to be within the specified directory. This Code is not supposed to by tidy, but include everything necessary for different visualizations of the result.
_______________
Created 20. May, 2026 by Gregor Bock

(0378 1735; ge27doc)

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

import Ausgelagerte_Funktionen_Versuchsauswertung as fkt

## Load Data
- Background field (and positions)
- Measured field (and positions)
- Simulated field (and positions)
- Insert zero field values of the QSpin manually!!!

<span style="color:red">The directions of the map and the zero field do not match!!!</span>
- MSR-x = QSpin-x
- MSR-y = QSpin-z
- MSR-z = QSpin-y

```dic_of_files_and_data``` is a dictionary ehich holds the Basename of all measurement files within the Key *File_Basename*. The dictionary also holds the field zero values of the QSpin sensor in the accouding array-entry in the dict-Key *field_zero_vals*. (Example: ```dic_of_files_and_data['File_Basename'][0][i]``` is the i-th Basename of a measurement file. By that it is also part of the filename of the simulation result, ```dic_of_files_and_data['field_zero_vals'][0][i]``` is an array of shape (3) holding the field zero value of the i-th measurment file in the QSpin coordinate system)

```Backgrounds``` has the same structure and is used for the Background-maps.

```folder_exp``` and ```folder_calc``` may need to be changed to find the files in the correct directories on **your** computer!

In [ ]:
dic_of_files_and_data = {'File_Basename': ['strom_10mA_03_2026-05-13_14-44-44', 'strom_5mA_04_2026-05-13_15-39-56', 'strom_05mA_05_2026-05-13_17-07-47', 'strom_0001A_02_2026-05-13_12-39-37', 'strom_10mA_mittel_07_2026-05-13_19-24-54', 'strom_5mA_mittel_08_2026-05-13_20-28-47', 'strom_05mA_mittel_09_2026-05-13_21-24-25', 'strom_0001A_mittel_06_2026-05-13_18-22-16', 'after_degauss_with_current_2026-04-27_16-54-25'],
                         'field_zero_vals': [[26472, 223, 5857], [-12479, -845, 3758], [-73, -1615, 1918], [1240, -1645, 1926], [-11305, -2671, 11], [-6347, -2203, 867], [-1849, -1749, 1689], [-2312, -1834, 1588], [0, 0, 0]]
                         }

Backgrounds = {'File_Basename': ['after_degauss_no_current_2026-04-27_16-07-03', 'background_01_2026-05-13_10-48-57'],
               'field_zero_vals': [[-1576, -1794, 1892], [-1541, -1741, 1764]]
               }

# Convert field zero values of measurement to MSR (Mapper) coordinate system and SI-units
for i, field_zero_vals in enumerate(dic_of_files_and_data['field_zero_vals']):
    # x-direction is correct -> only switch y and z (positiv sign, if the QSpin is inserted to the mapper with the cable at the upper side)
    B_0_x_QSpin = field_zero_vals[0]
    B_0_y_QSpin = field_zero_vals[1]
    B_0_z_QSpin = field_zero_vals[2]
    dic_of_files_and_data['field_zero_vals'][i] = [B_0_x_QSpin * 10**(-12), -B_0_z_QSpin * 10**(-12), -B_0_y_QSpin * 10**(-12)]

# Convert field zero values of Background to MSR (Mapper) coordinate system and SI-units
for i, field_zero_vals in enumerate(Backgrounds['field_zero_vals']):
    # x-direction is correct -> only switch y and z (positiv sign, if the QSpin is inserted to the mapper with the cable at the upper side)
    B_0_x_QSpin = field_zero_vals[0]
    B_0_y_QSpin = field_zero_vals[1]
    B_0_z_QSpin = field_zero_vals[2]
    Backgrounds['field_zero_vals'][i] = [B_0_x_QSpin * 10**(-12), -B_0_z_QSpin * 10**(-12), -B_0_y_QSpin * 10**(-12)]

# Give local paths as well as prefix and suffix for file names
folder_exp = "D:\\Studium\\Physik\\Bachelorarbeit\\MSR-field_cancelation\\Experiments\\"
folder_calc = "D:\\Studium\\Physik\\Bachelorarbeit\\MSR-field_cancelation\\B_tot_results\\"
prefix_calc = "B_tot_result_" # Change this when ready
suffix = "\\map\\points"

# If .npz file does not hold geometric track data, specify it here
L_x = 0.800                                         # length of the mapped volume in x-direction
L_y = 0.800                                         # length of the mapped volume in y-direction
L_z = 0.400                                         # length of the mapped volume in z-direction
step_size = 0.200                                   # size of the grid steps

shift_x = 0                                         # shift of the mapped volume in x-direction
shift_y = 0                                         # shift of the mapped volume in y-direction
shift_z = 0                                         # shift of the mapped volume in z-direction

# Loading Background data:
Background_map = 1                                  # Specify which background file shall be loaded
folder_path_background = folder_exp + Backgrounds['File_Basename'][Background_map] + suffix

print(f'\nStarting to load data for background file: {Backgrounds["File_Basename"][Background_map]}')

target_point_coord_exp, B_target_point_background, _ = fkt.load_data_from_folder(folder_path_background, folder_path_background, L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)
for comp in range(3):
    B_target_point_background[:, comp] = B_target_point_background[:, comp] - Backgrounds['field_zero_vals'][Background_map][comp]  # Subtract field zero values from B-fields

# Loading measurement data:
B_target_point = {'File_Basename': [], 'B_field_exp': [], 'B_field_calc': [],'target_point_coord_calc': [], 'stream_func': [], 'd_coil': [], 'current': [], 'n_windings': [], 'all_coils': []}

for idx, Basename in enumerate(dic_of_files_and_data['File_Basename'][0:4]):
    
    print(f'\nStarting to load data for measurement file: {Basename}')
    B_target_point['File_Basename'].append(Basename)

    folder_path_experiment = folder_exp + Basename + suffix
    folder_path_calc = folder_calc + prefix_calc + Basename + '.npz'
    
    # Load data using the function "load_data_from_folder" from the "Ausgelagerte_Funktionen_Versuchsauswertung.py" file.
    B_target_point['B_field_exp'].append(fkt.load_data_from_folder(folder_path_background, folder_path_experiment, L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)[2])
    for comp in range(3):
        B_target_point['B_field_exp'][idx][:, comp] = B_target_point['B_field_exp'][idx][:, comp] - dic_of_files_and_data['field_zero_vals'][idx][comp]

    # Load data of calculation
    data_sim = np.load(folder_path_calc)
    # Debug
    # print(f'The names of the arrays of the .npz-file are: {sorted(data_sim.files)}')  # prints all available keys in data_sim

    B_target_point['B_field_calc'].append(data_sim['B_field'])
    B_target_point['target_point_coord_calc'].append(data_sim['points'])
    B_target_point['stream_func'].append(data_sim['stream_function'])
    B_target_point['d_coil'].append(data_sim['d_coil'])
    B_target_point['current'].append(data_sim['current'])
    B_target_point['n_windings'].append(data_sim['n_windings'])
    B_target_point['all_coils'].append(data_sim['all_coils'])

## Some simple plots :)

In [ ]:
# Plot the coil layup, such that it is clear, which experiment setup was used. Additionally, print coil-diameter, currrent and windings, since they can not be determined by the plot alone

for idx, all_coils in enumerate(B_target_point['all_coils']):
    fig_coil_layup = plt.figure()
    ax_coil_layup = fig_coil_layup.add_subplot(111, projection='3d')
    for k in range(all_coils.shape[0]):
        ax_coil_layup.plot(all_coils[k, :, 0], all_coils[k, :, 1], all_coils[k, :, 2], color='blue')
    ax_coil_layup.set_xlabel(r'$x$')
    ax_coil_layup.set_ylabel(r'$y$')
    ax_coil_layup.set_zlabel(r'$z$')
    plt.show()

    print(f'The coil diameter used in the simulation is ' + r"$d_{coil}$ =" + f' {B_target_point['d_coil'][idx]} m')
    print(f'The coil current used in the simulation is ' + r"$I_{coil}$ =" + f' {B_target_point['current'][idx]} A')
    print(f'The number of windings of each coil used in the simulation is ' + r"$n_{windings}$ =" + f' {B_target_point['n_windings'][idx]}')

## Plot magnetic fields (norm)

This should give some intuition on how to access and work with the given arrays. Moreover a slight intuition for the field can be obtained (maybe)

In [ ]:
# Plot total field from Background (without coil currents)
fig_B_background = plt.figure()
ax_B_background = fig_B_background.add_subplot(111, projection='3d')
sc_B_background = ax_B_background.scatter(
    target_point_coord_exp[:, 0],
    target_point_coord_exp[:, 1],
    target_point_coord_exp[:, 2],
    c = np.linalg.norm(B_target_point_background, axis=1),
    s = 100,
    cmap = 'viridis',
    alpha = 0.5,
)
fig_B_background.colorbar(sc_B_background, ax=ax_B_background, label=r'$|\mathbf{B}_{\text{background}}|$')
name = Backgrounds['File_Basename'][Background_map]
ax_B_background.set_title(f'Residual B-field without any currents of\n{name}')
ax_B_background.set_xlabel(r'$x$')
ax_B_background.set_ylabel(r'$y$')

B_calc_coarse = []
for idx in range(len(B_target_point['File_Basename'])):
    B_calc = B_target_point['B_field_calc'][idx]
    B_exp = B_target_point['B_field_exp'][idx]
    target_point_coord_calc = B_target_point['target_point_coord_calc'][idx]
    name = B_target_point['File_Basename'][idx]

    # Plot total field from simulation
    fig_B_tot = plt.figure()
    ax_B_tot = fig_B_tot.add_subplot(111, projection='3d')
    sc_B_tot = ax_B_tot.scatter(
        target_point_coord_calc[:, 0],
        target_point_coord_calc[:, 1],
        target_point_coord_calc[:, 2],
        c = np.linalg.norm(B_calc, axis=1),
        s = 100,
        cmap = 'viridis',
        alpha = 0.5,
    )
    fig_B_tot.colorbar(sc_B_tot, ax=ax_B_tot, label=r'$|\mathbf{B}_{\text{total}}|$')
    ax_B_tot.set_title(f'Simulation results B-field of\n{name}')
    ax_B_tot.set_xlabel(r'$x$')
    ax_B_tot.set_ylabel(r'$y$')

    # These values are specified in the Versuch_1_Auswertung-file and need to be redifined here ):
    coil_plane_dist_to_origin_x = 2.34/2
    coil_plane_dist_to_origin_y = 2.34/2
    coil_plane_dist_to_origin_z = 2.20/2
    safety_distance = 0.05

    # Plot coarsened field from simulation for better comparison
    num_calc_target_points_fine = int( (len(target_point_coord_calc) + 1) **(1/3) )
    B_tot_coarse = fkt.interpolate_B_on_coarse_grid(num_calc_target_points_fine, target_point_coord_exp, coil_plane_dist_to_origin_x, coil_plane_dist_to_origin_y, coil_plane_dist_to_origin_z, safety_distance, B_calc)
    B_calc_coarse.append(B_tot_coarse)

    fig_B_tot_coarse = plt.figure()
    ax_B_tot_coarse = fig_B_tot_coarse.add_subplot(111, projection='3d')
    sc_B_tot_coarse = ax_B_tot_coarse.scatter(
        target_point_coord_exp[:, 0],
        target_point_coord_exp[:, 1],
        target_point_coord_exp[:, 2],
        c = np.linalg.norm(B_tot_coarse, axis=1),
        s = 100,
        cmap = 'viridis',
        alpha = 0.5,
    )
    fig_B_tot_coarse.colorbar(sc_B_tot_coarse, ax=ax_B_tot_coarse, label=r'$|\mathbf{B}_{\text{total}}|$')
    ax_B_tot_coarse.set_title(f'Simulated results B-field of\n{name}')
    ax_B_tot_coarse.set_xlabel(r'$x$')
    ax_B_tot_coarse.set_ylabel(r'$y$')


    # Plot total field from Experiment (with coil currents)
    fig_B_exp = plt.figure()
    ax_B_exp = fig_B_exp.add_subplot(111, projection='3d')
    sc_B_exp = ax_B_exp.scatter(
        target_point_coord_exp[:, 0],
        target_point_coord_exp[:, 1],
        target_point_coord_exp[:, 2],
        c = np.linalg.norm(B_exp, axis=1),
        s = 100,
        cmap = 'viridis',
        alpha = 0.5,
    )
    fig_B_exp.colorbar(sc_B_exp, ax=ax_B_exp, label=r'$|\mathbf{B}_{\text{experiment}}|$')
    ax_B_exp.set_title(f'B-field from experiment\n{name}')
    ax_B_exp.set_xlabel(r'$x$')
    ax_B_exp.set_ylabel(r'$y$') 

    plt.show()

## Differences of fields and their statistical behaviour

Here the difference of the computed and measured field shall be shown. This should be as small as possible!

<span style="color:red">**Note** that the directions of the field until now do not have to be he same. If a ```+``` or a ```-``` is used must be evaluated manually!!!</span>

In [ ]:
B_diff_arr = []
for idx in range(len(B_target_point['File_Basename'])):
    B_calc = B_target_point['B_field_calc'][idx]
    B_exp = B_target_point['B_field_exp'][idx]

    B_calc_av = np.average(B_calc)
    B_exp_av = np.average(B_exp)
    if B_calc_av - B_exp_av < B_calc_av + B_exp_av:
        B_diff = B_calc_coarse[idx] - B_exp
    else:
        B_diff = B_calc_coarse[idx] + B_exp

    B_diff_arr.append(B_diff)
    B_diff_norm = np.linalg.norm(B_diff, axis=1)

    fig_B_diff = plt.figure()
    ax_B_diff = fig_B_diff.add_subplot(111, projection='3d')
    sc_B_diff = ax_B_diff.scatter(
        target_point_coord_exp[:, 0],
        target_point_coord_exp[:, 1],
        target_point_coord_exp[:, 2],
        c = B_diff_norm,
        s = 100,
        cmap = 'viridis',
        alpha = 0.5,
    )
    fig_B_diff.colorbar(sc_B_diff, ax=ax_B_diff, label=r'$|\mathbf{B}_{\text{difference}}|$')
    ax_B_diff.set_title(f'Difference between measured and simulated B-field of\n{B_target_point['File_Basename'][idx]}')
    ax_B_diff.set_xlabel(r'$x$')
    ax_B_diff.set_ylabel(r'$y$')

    # Calculate statistical measures
    B_diff_norm = np.linalg.norm(B_diff, axis=1) * 10**9
    B_diff_variance = np.var(B_diff_norm)
    B_diff_std = np.sqrt(B_diff_variance)
    B_diff_average = np.average(B_diff_norm)

    # Create Histogramm of differences
    fig_B_diff_hist = plt.figure()
    ax_B_diff_hist = fig_B_diff_hist.add_subplot()
    ax_B_diff_hist.hist(B_diff_norm, 32, alpha = 0.8)

    counts, bins, _ = ax_B_diff_hist.hist(B_diff_norm, bins=32, alpha=0.6)

    x = np.linspace(min(B_diff_norm), max(B_diff_norm), 200)
    bin_width = bins[1] - bins[0]
    Gauss = (1 / (B_diff_std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - B_diff_average) / B_diff_std) ** 2)
    Gauss_scaled = Gauss * len(B_diff_norm) * bin_width

    ax_B_diff_hist.plot(x, Gauss_scaled, 'g')

    ax_B_diff_hist.set_title(f'Histogramm of difference of measurement and simulation\n{B_target_point['File_Basename'][idx]}')
    ax_B_diff_hist.set_xlabel(f'Difference between simulated and measured B-field in nT')
    ax_B_diff_hist.set_ylabel(f'Number of points with described difference')

    plt.show()

    print(f'{B_target_point['File_Basename'][idx]}\n   The average value of the difference in the norm of the B-field is {B_diff_average:.2f} nT\n   The standard deviation of the difference in the norm of the B-field is {B_diff_std:.2f} nT')

## Field-strength along coordinate axis

In [ ]:
fig_Bx_Xaxis = plt.figure()
ax_Bx_Xaxis = fig_Bx_Xaxis.add_subplot()
ax_Bx_Xaxis.set_title(r'$B_x$ along the $x$-axis')

fig_By_Xaxis = plt.figure()
ax_By_Xaxis = fig_By_Xaxis.add_subplot()
ax_By_Xaxis.set_title(r'$B_y$ along the $x$-axis')

fig_Bz_Xaxis = plt.figure()
ax_Bz_Xaxis = fig_Bz_Xaxis.add_subplot()
ax_Bz_Xaxis.set_title(r'$B_z$ along the $x$-axis')

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

for idx in range(len(B_target_point['File_Basename'])):
    B_calc_x = B_target_point['B_field_calc'][idx][:, 0]
    B_calc_y = B_target_point['B_field_calc'][idx][:, 1]
    B_calc_z = B_target_point['B_field_calc'][idx][:, 2]
    target_coord_calc = B_target_point['target_point_coord_calc'][idx]

    Bx_calc_x_axis = []
    By_calc_x_axis = []
    Bz_calc_x_axis = []
    By_calc_y_axis = []
    Bz_calc_z_axis = []
    Coord_calc_x_axis = []
    Coord_calc_y_axis = []
    Coord_calc_z_axis = []
    for p_idx, point in enumerate(target_coord_calc):
        if point[1] == 0 and point[2] == 0 and np.abs(point[0]) < np.max(target_point_coord_exp[:, 0]):             # Point is located on x-axis
            Bx_calc_x_axis.append(B_calc_x[p_idx])
            By_calc_x_axis.append(B_calc_y[p_idx])
            Bz_calc_x_axis.append(B_calc_z[p_idx])
            Coord_calc_x_axis.append(point[0])
        else:
            pass
        if point[0] == 0 and point[2] == 0 and np.abs(point[1]) < np.max(target_point_coord_exp[:, 1]):             # Point is located on y-axis
            By_calc_y_axis.append(B_calc_y[p_idx])
            Coord_calc_y_axis.append(point[1])
        else:
            pass
        if point[0] == 0 and point[1] == 0 and np.abs(point[2]) < np.max(target_point_coord_exp[:, 2]):             # Point is located on z-axis
            Bz_calc_z_axis.append(B_calc_z[p_idx])
            Coord_calc_z_axis.append(point[2])
        else:
            pass

    B_exp_x = B_target_point['B_field_exp'][idx][:, 0]
    B_exp_y = B_target_point['B_field_exp'][idx][:, 1]
    B_exp_z = B_target_point['B_field_exp'][idx][:, 2]
    target_coord_exp = target_point_coord_exp

    Bx_exp_x_axis = []
    By_exp_x_axis = []
    Bz_exp_x_axis = []
    By_exp_y_axis = []
    Bz_exp_z_axis = []
    Coord_exp_x_axis = []
    Coord_exp_y_axis = []
    Coord_exp_z_axis = []
    for p_idx, point in enumerate(target_coord_exp):
        if point[1] == 0 and point[2] == 0:             # Point is located on x-axis
            Bx_exp_x_axis.append(B_exp_x[p_idx])
            By_exp_x_axis.append(B_exp_y[p_idx])
            Bz_exp_x_axis.append(B_exp_z[p_idx])
            Coord_exp_x_axis.append(point[0])
        else:
            pass
        if point[0] == 0 and point[2] == 0:             # Point is located on y-axis
            By_exp_y_axis.append(B_exp_y[p_idx])
            Coord_exp_y_axis.append(point[1])
        else:
            pass
        if point[0] == 0 and point[1] == 0:             # Point is located on z-axis
            Bz_exp_z_axis.append(B_exp_z[p_idx])
            Coord_exp_z_axis.append(point[2])
        else:
            pass
    
    color = colors[idx % len(colors)]
    ax_Bx_Xaxis.plot(Coord_exp_x_axis, np.abs(Bx_exp_x_axis), ls = '-', color = color, label = f"Exp.: {B_target_point['File_Basename'][idx]}")
    ax_Bx_Xaxis.plot(Coord_calc_x_axis, np.abs(Bx_calc_x_axis), ls = '--', color = color, label = f"Calc.: {B_target_point['File_Basename'][idx]}")

    # ax_By_Xaxis.plot(Coord_exp_x_axis, By_exp_x_axis, ls = '-', color = color, label = f"Exp.: {B_target_point['File_Basename'][idx]}")
    ax_By_Xaxis.plot(Coord_calc_x_axis, By_calc_x_axis, ls = '--', color = color, label = f"Calc.: {B_target_point['File_Basename'][idx]}")

    # ax_Bz_Xaxis.plot(Coord_exp_x_axis, Bz_exp_x_axis, ls = '-', color = color, label = f"Exp.: {B_target_point['File_Basename'][idx]}")
    ax_Bz_Xaxis.plot(Coord_calc_x_axis, Bz_calc_x_axis, ls = '--', color = color, label = f"Calc.: {B_target_point['File_Basename'][idx]}")

ax_Bx_Xaxis.grid()
ax_Bx_Xaxis.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
ax_Bx_Xaxis.set_xlabel(r'$x$')
ax_Bx_Xaxis.set_ylabel(r'$B_x$')

ax_By_Xaxis.grid()
ax_By_Xaxis.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
ax_By_Xaxis.set_xlabel(r'$x$')
ax_By_Xaxis.set_ylabel(r'$B_y$')

ax_Bz_Xaxis.grid()
ax_Bz_Xaxis.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
ax_Bz_Xaxis.set_xlabel(r'$x$')
ax_Bz_Xaxis.set_ylabel(r'$B_z$')

plt.subplots_adjust(bottom=0.25)

plt.show()